# Glioblastoma Brain Tumor Segmentation — BraTS 2020 (3D U-Net)

*Companion notebook to the bachelor thesis "Detection and Classification of Glioblastoma Brain Tumor".*

This notebook trains and evaluates a **3D U-Net** for multi-class segmentation of glioblastoma
sub-regions in multi-modal brain MRI, using the **BraTS 2020** dataset.

**Pipeline:** load 4 MRI modalities (T1, T1ce, T2, FLAIR) → 3D U-Net → 4-class voxel segmentation
→ evaluate with Dice + HD95 → visual overlays.

| Label | Region | Raw BraTS value |
|-------|--------|-----------------|
| 0 | Background / healthy | 0 |
| 1 | Necrotic & non-enhancing tumor core (NCR/NET) | 1 |
| 2 | Peritumoral edema (ED) | 2 |
| 3 | Enhancing tumor (ET) | 4 (remapped → 3) |

**On scope (glioblastoma vs. glioma):** BraTS 2020's 369 subjects are labelled by grade in
`name_mapping.csv` — **293 HGG** (high-grade glioma = glioblastoma, WHO grade IV) and **76 LGG**
(lower-grade glioma). Set `GRADE_FILTER = "HGG"` below to train and evaluate on genuine
glioblastoma cases only, which keeps the thesis's "glioblastoma" framing accurate.

---
### How to run
1. Make sure the project dependencies are installed (`pip install -r requirements.txt`) and that
   this notebook runs on that environment's kernel.
2. Run the cells top to bottom.
3. **First pass:** leave `QUICK_TEST = True` — it uses a small subset and 2 epochs to confirm the
   whole notebook runs end-to-end in a few minutes. Then set `QUICK_TEST = False` and increase
   `MAX_EPOCHS` for the real training run that produces the thesis results.

> **GPU note:** 3D training is heavy. The config below is tuned for an 8 GB GPU (128³ patches,
> batch size 1, mixed precision). If you hit a CUDA out-of-memory error, reduce `ROI_SIZE`
> (e.g. `(96, 96, 96)`). Real GPU training requires the CUDA build of PyTorch — the environment
> check cell reports whether CUDA is available.

## 1. Setup & configuration

In [ ]:
import os
import sys
from pathlib import Path

# Make the project package importable (assumes the notebook sits in the repo root).
REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ============================== CONFIG ==============================
# Path to the BraTS2020 training data (the folder that directly contains
# the BraTS20_Training_XXX subject subfolders + name_mapping.csv).
DATA_ROOT = r"U:\BraTS2020_TrainingData\MICCAI_BraTS2020_TrainingData"

GRADE_FILTER    = "HGG"          # "HGG" (glioblastoma), "LGG", or None (all 369 subjects)
ROI_SIZE        = (128, 128, 128)  # 3D patch size (D, H, W); reduce if GPU OOM
BATCH_SIZE      = 1
MAX_EPOCHS      = 50             # increase (e.g. 100-150) for final thesis results
LEARNING_RATE   = 1e-4
TRAIN_VAL_SPLIT = 0.85
SEED            = 42
DEVICE          = "auto"         # "auto" | "cuda" | "cpu"
NUM_WORKERS     = 0              # keep 0 on Windows/Jupyter to avoid loader issues

# Quick end-to-end validation run (small subset, few epochs).
QUICK_TEST          = True
QUICK_TEST_SUBJECTS = 12
QUICK_TEST_EPOCHS   = 2

# Outputs: checkpoints are large (~230 MB each) -> keep them off the system drive.
CHECKPOINT_DIR = Path(r"U:\brats_outputs\checkpoints")
OUTPUT_DIR     = REPO_ROOT / "outputs"      # figures + metrics for the thesis (small)
EXPERIMENT_NAME = f"brats2020_{(GRADE_FILTER or 'all').lower()}"
# ===================================================================

EPOCHS = QUICK_TEST_EPOCHS if QUICK_TEST else MAX_EPOCHS
print("Repo root       :", REPO_ROOT)
print("Data root exists:", Path(DATA_ROOT).exists())
print("Grade filter    :", GRADE_FILTER)
print("Epochs          :", EPOCHS, "(QUICK_TEST)" if QUICK_TEST else "")

In [ ]:
import torch
import monai

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory/1e9:.1f} GB)")
else:
    print("No CUDA GPU detected -> training will run on CPU (very slow for 3D).")
print("MONAI:", monai.__version__)

## 2. Data loading & inspection

Each subject provides four co-registered MRI modalities and an expert segmentation. The loader
(`NiftiBraTSDataset`) stacks the modalities into a `(4, D, H, W)` tensor, z-score normalizes each
modality, remaps the segmentation label `4 → 3`, and center-crops to `ROI_SIZE`. It also handles
the known BraTS2020 quirk where subject 355's mask has a non-standard filename, and can filter by
tumor grade using `name_mapping.csv`.

In [ ]:
from omegaconf import OmegaConf
from torch.utils.data import Subset

cfg = OmegaConf.create({
    "seed": SEED, "task": "segmentation", "experiment_name": EXPERIMENT_NAME,
    "paths": {"data_root": DATA_ROOT, "output_dir": str(OUTPUT_DIR),
              "checkpoint_dir": str(CHECKPOINT_DIR)},
    "data": {"mode": "nifti", "roi_size": list(ROI_SIZE), "batch_size": BATCH_SIZE,
             "num_workers": NUM_WORKERS, "train_val_split": TRAIN_VAL_SPLIT,
             "grade_filter": GRADE_FILTER},
    "model": {"name": "unet", "in_channels": 4, "out_channels": 4, "dropout": 0.2},
    "training": {"device": DEVICE, "max_epochs": EPOCHS, "learning_rate": LEARNING_RATE,
                 "weight_decay": 1e-4, "val_interval": 1, "ckpt_interval": 10,
                 "scheduler": "cosine", "early_stopping_patience": None},
})

from data import build_dataset

train_ds = build_dataset(cfg, "train")
val_ds   = build_dataset(cfg, "val")
print(f"Full split -> train: {len(train_ds)} subjects | val: {len(val_ds)} subjects")

if QUICK_TEST:
    k_tr = min(QUICK_TEST_SUBJECTS, len(train_ds))
    k_va = max(1, min(QUICK_TEST_SUBJECTS // 4, len(val_ds)))
    train_ds = Subset(train_ds, list(range(k_tr)))
    val_ds   = Subset(val_ds, list(range(k_va)))
    print(f"[QUICK_TEST] using {len(train_ds)} train / {len(val_ds)} val subjects")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualize one subject: the four modalities + the ground-truth tumor overlay.
sample = train_ds[0]
img = sample["image"].numpy()      # (4, D, H, W)
msk = sample["mask"].numpy()[0]    # (D, H, W)

tumor_per_slice = (msk > 0).reshape(msk.shape[0], -1).sum(1)
s = int(tumor_per_slice.argmax()) if tumor_per_slice.max() > 0 else msk.shape[0] // 2

mod_names = ["T1", "T1ce", "T2", "FLAIR"]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i in range(4):
    axes[i].imshow(img[i, s], cmap="gray")
    axes[i].set_title(mod_names[i]); axes[i].axis("off")
axes[4].imshow(img[3, s], cmap="gray")
axes[4].imshow(np.ma.masked_where(msk[s] == 0, msk[s]), cmap="jet", alpha=0.5)
axes[4].set_title("FLAIR + tumor mask"); axes[4].axis("off")
plt.suptitle(f"Sample subject — axial slice {s}")
plt.tight_layout(); plt.show()

## 3. Model — 3D U-Net

A 3D U-Net (MONAI) with an encoder–decoder structure and skip connections. The encoder downsamples
through channels `(32, 64, 128, 256, 512)` with residual units and batch normalization; the decoder
upsamples symmetrically and fuses encoder features via skip connections to recover voxel-level
detail. Input: 4 modality channels; output: 4 class logits per voxel.

In [ ]:
from models.build_model import build_model
from models.segmentation import get_segmentation_channels_from_dataset

in_ch, out_ch = get_segmentation_channels_from_dataset(train_ds)
model = build_model(cfg, in_channels=in_ch, out_channels=out_ch)
n_params = sum(p.numel() for p in model.parameters())
print(f"3D U-Net | in_channels={in_ch} | out_channels={out_ch} | parameters={n_params:,}")

## 4. Training

- **Loss:** Dice + Cross-Entropy (`DiceCELoss`) — combines region overlap with per-voxel classification.
- **Optimizer:** AdamW (`lr=1e-4`, weight decay `1e-4`).
- **Scheduler:** cosine annealing.
- **Mixed precision** (AMP) is used automatically on CUDA to reduce memory.
- Augmentation: random flips + 90° rotations, with a pad/crop guaranteeing a fixed patch size.

The best checkpoint (highest validation mean Dice) is saved to `CHECKPOINT_DIR`.

In [ ]:
from torch.utils.data import DataLoader
from training.transforms import build_transforms
from training.dataset_wrappers import TransformedDataset
from training.training import Trainer
from models.loss import get_dice_ce_loss
from utils.seed import set_seed

set_seed(SEED)
device = torch.device("cuda" if (DEVICE != "cpu" and torch.cuda.is_available()) else "cpu")
print("Training on:", device)

train_tf  = build_transforms(cfg, "train")
train_aug = TransformedDataset(train_ds, train_tf)

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_aug, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"), generator=g)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)

trainer = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    loss_fn=get_dice_ce_loss(num_classes=out_ch), task="segmentation",
    output_dir=OUTPUT_DIR, checkpoint_dir=CHECKPOINT_DIR, experiment_name=EXPERIMENT_NAME,
    device=device, learning_rate=LEARNING_RATE, max_epochs=EPOCHS,
    scheduler="cosine", early_stopping_patience=None, num_classes=out_ch, progress=True,
)
result = trainer.train()
print(f"\nBest {trainer.metric_key} = {result['best_metric']:.4f} at epoch {result['best_epoch']}")

In [ ]:
# Training curves (loss and validation Dice)
h = result["history"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h["epoch"], h["train_loss"], label="train loss")
ax[0].plot(h["epoch"], h["val_loss"], label="val loss")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(h["epoch"], h["mean_dice"], color="green", marker="o", label="val mean Dice")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("Dice"); ax[1].set_title("Validation mean Dice")
ax[1].legend()
plt.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved:", OUTPUT_DIR / "training_curves.png")

## 5. Evaluation — Dice, HD95 & overlays

We reload the best checkpoint and evaluate on the validation set:
- **Dice** per class (region overlap; higher is better).
- **HD95** per class (95th-percentile Hausdorff distance, in voxels; boundary error, lower is better).
- **Overlay panels** (MRI | ground truth | prediction) saved for the thesis figures.

In [ ]:
from evaluation.segmentation_eval import evaluate_segmentation

best_ckpt = CHECKPOINT_DIR / f"{EXPERIMENT_NAME}_best.pt"
ck = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(ck["model_state_dict"])
print(f"Loaded best checkpoint (epoch {ck.get('epoch', '?')})")

metrics = evaluate_segmentation(
    model, val_loader, device, num_classes=out_ch,
    out_dir=OUTPUT_DIR / "segmentation_eval", num_overlays=6,
)

print("\n=== Segmentation metrics (validation) ===")
print(f"{'class':<12}{'Dice':>10}{'HD95':>12}")
for name in metrics["dice_per_class"]:
    d = metrics["dice_per_class"][name]
    hd = metrics["hd95_per_class"][name]
    d_s = f"{d:.4f}" if d is not None else "n/a"
    hd_s = f"{hd:.2f}" if hd is not None else "n/a"
    print(f"{name:<12}{d_s:>10}{hd_s:>12}")
print("-" * 34)
print(f"{'mean (fg)':<12}{metrics['mean_dice_foreground']:.4f}"
      f"{metrics['mean_hd95_foreground']:>12.2f}")
print("\nFigures + metrics.json ->", OUTPUT_DIR / "segmentation_eval")

In [ ]:
# Display the saved overlay figures inline
import glob
from IPython.display import Image as IPyImage, display

overlays = sorted(glob.glob(str(OUTPUT_DIR / "segmentation_eval" / "overlays" / "*.png")))
print(f"{len(overlays)} overlay figure(s):")
for p in overlays:
    display(IPyImage(filename=p))

## 6. Inference on an unlabeled validation volume (optional)

The BraTS 2020 *validation* set has no public ground-truth masks. Here we run the trained model on
one validation subject purely to visualize the predicted tumor segmentation on unseen data. Set the
path below if your validation data is elsewhere.

In [ ]:
import nibabel as nib

VAL_ROOT = Path(r"U:\BraTS2020_ValidationData\MICCAI_BraTS2020_ValidationData")

def _load_nii(path_wo_ext):
    for ext in (".nii", ".nii.gz"):
        p = Path(str(path_wo_ext) + ext)
        if p.exists():
            return nib.load(p).get_fdata()
    raise FileNotFoundError(path_wo_ext)

def _center_crop(v, roi):
    _, d, h, w = v.shape; pd, ph, pw = roi
    d0, h0, w0 = max(0, d//2 - pd//2), max(0, h//2 - ph//2), max(0, w//2 - pw//2)
    return v[:, d0:d0+pd, h0:h0+ph, w0:w0+pw]

if VAL_ROOT.exists():
    subj = sorted(p for p in VAL_ROOT.iterdir() if p.is_dir())[0]
    pid = subj.name
    vol = np.stack([_load_nii(subj / f"{pid}_{m}") for m in ["t1", "t1ce", "t2", "flair"]], 0).astype(np.float32)
    mu = vol.mean((-3, -2, -1), keepdims=True); sd = vol.std((-3, -2, -1), keepdims=True); sd[sd < 1e-8] = 1
    vol = (vol - mu) / sd
    vc = _center_crop(vol, ROI_SIZE)

    model.eval()
    with torch.no_grad():
        pred = model(torch.from_numpy(vc).unsqueeze(0).to(device)).argmax(1)[0].cpu().numpy()

    tps = (pred > 0).reshape(pred.shape[0], -1).sum(1)
    s = int(tps.argmax()) if tps.max() > 0 else pred.shape[0] // 2
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    ax[0].imshow(vc[3, s], cmap="gray"); ax[0].set_title(f"{pid} — FLAIR"); ax[0].axis("off")
    ax[1].imshow(vc[3, s], cmap="gray")
    ax[1].imshow(np.ma.masked_where(pred[s] == 0, pred[s]), cmap="jet", alpha=0.5)
    ax[1].set_title("Predicted tumor"); ax[1].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Validation data not found at", VAL_ROOT, "- skipping inference demo.")

## 7. Notes & next steps

- **For final thesis results:** set `QUICK_TEST = False` and `MAX_EPOCHS` to ~100–150, then re-run
  from section 4. Training on the full HGG set on an 8 GB GPU takes several hours — run it overnight.
- **All thesis figures are saved under `outputs/`:** `training_curves.png`, and
  `segmentation_eval/` (per-class `metrics.json` + overlay panels).
- **Scope:** with `GRADE_FILTER="HGG"` the model is trained and evaluated on high-grade glioma
  (glioblastoma) only — report this in the methodology so the "glioblastoma" claim is precise.
- **If you hit CUDA OOM:** lower `ROI_SIZE` to `(96, 96, 96)`, keep `BATCH_SIZE = 1`.
- **Possible extensions:** slice-level tumor *detection* (a confusion matrix derived from the
  segmentation, matching the original report's Chapter 4), survival-day regression (the dataset ships
  `survival_info.csv` with age / survival / resection), and a deployable web app for inference.